In [ ]:
import re 
import os
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
import torch


In [ ]:

if torch.cuda.is_available():    

    device = torch.device("cuda")

    print('There are %d GPU(s) available.' % torch.cuda.device_count())

    print('We will use the GPU:', torch.cuda.get_device_name(0))

else:
    print('No GPU available, using the CPU instead.')
    device = torch.device("cpu")

In [ ]:
!pip install transformers

In [ ]:
csv_path = r"C:\Users\FPTSHOP\OneDrive\Máy tính\data_hatespeech_text\sentiment_text\training.1600000.processed.noemoticon.csv"
df =  pd.read_csv(csv_path, encoding="ISO-8859-1", header=None , names=['label', 'ids', 'data', 'flag' , 'user','sentence'])

# Report the number of sentences.
print('Number of training sentences: {:,}\n'.format(df.shape[0]))

# Display 10 random rows from the data.
df.sample(10)

In [ ]:
df = df[['label' ,'sentence']]
df['label'] = df['label'].replace({4: 1} )
df

In [ ]:
label_counts = df['label'].value_counts()
print(label_counts)

In [ ]:
urlPattern        = r"((http://)[^ ]*|(https://)[^ ]*|(www\.)[^ ]*)"
userPattern       = '@[^\s]+'
hashtagPattern    = '#[^\s]+'
sequencePattern   = r"(.)\1\1+"
seqReplacePattern = r"\1\1"


def preprocess_apply(tweet):

    tweet = tweet.lower()

    # Replace all URls with '<url>'
    tweet = re.sub(urlPattern,'',tweet)
    # Replace @USERNAME to '<user>'.
    tweet = re.sub(userPattern,'', tweet)
    
    # Replace 3 or more consecutive letters by 2 letter.
    tweet = re.sub(sequencePattern, seqReplacePattern, tweet)

    # Adding space on either side of '/' to seperate words (After replacing URLS).
    tweet = re.sub(r'/', ' / ', tweet)
    return tweet

In [ ]:
df['preprocessing_sentence'] = df['sentence'].apply(preprocess_apply)
df

In [ ]:
df['word_count'] = df['preprocessing_sentence'].apply(lambda x: len(str(x).split()))

# Calculate the statistics
plt.figure(figsize=(8, 6))
sns.violinplot(x=df['word_count'])
plt.title('Violin Plot of Word Count per Tweet')
plt.xlabel('Word Count')
plt.show()

In [ ]:
df

In [ ]:
sentences = df.preprocessing_sentence.values
labels = df.label.values

In [ ]:
from transformers import BertTokenizer

# Load the BERT tokenizer.
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

In [ ]:
# Print the original sentence.
print(' Original: ', sentences[0])

# Print the sentence split into tokens.
print('Tokenized: ', tokenizer.tokenize(sentences[0]))

# Print the sentence mapped to token ids.
print('Token IDs: ', tokenizer.convert_tokens_to_ids(tokenizer.tokenize(sentences[0])))

In [ ]:
# max_len = 0
# total_len = 0

# for sent in sentences:
#     input_ids = tokenizer.encode(sent, add_special_tokens=True)
#     max_len = max(max_len, len(input_ids))
#     total_len += len(input_ids)

# # Calculate the average sentence length
# average_len = total_len / len(sentences)

# print('Max sentence length:', max_len)
# print('Average sentence length:', average_len)

In [ ]:
input_ids = []
attention_masks = []

for sent in sentences:

    encoded_dict = tokenizer.encode_plus(
                        sent,                      # Sentence to encode.
                        add_special_tokens = True, # Add '[CLS]' and '[SEP]'
                        max_length = 64,           # Pad & truncate all sentences.
                        pad_to_max_length = True,
                        truncation=True,
                        return_attention_mask = True,   # Construct attn. masks.
                        return_tensors = 'pt',     # Return pytorch tensors.
                   )
    
    input_ids.append(encoded_dict['input_ids'])
    attention_masks.append(encoded_dict['attention_mask'])


input_ids = torch.cat(input_ids, dim=0)
attention_masks = torch.cat(attention_masks, dim=0)
labels = torch.tensor(labels)

# Print sentence 0, now as a list of IDs.
print('Original: ', sentences[0])
print('Token IDs:', input_ids[0])

In [ ]:
from torch.utils.data import TensorDataset, random_split

dataset = TensorDataset(input_ids, attention_masks, labels)

# Define split sizes
train_size = int(0.9 * len(dataset)) 
val_size = int(0.07 * len(dataset))    
test_size = len(dataset) - train_size - val_size  



train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

# Output the sizes of each dataset
print(f'{train_size:,} training samples')
print(f'{val_size:,} validation samples')
print(f'{test_size:,} test samples')


In [ ]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler


batch_size = 128


train_dataloader = DataLoader(
            train_dataset,
            sampler = RandomSampler(train_dataset),
            batch_size = batch_size 
        )


validation_dataloader = DataLoader(
            val_dataset,
            sampler = SequentialSampler(val_dataset), 
            batch_size = batch_size 
        )

test_dataloader = DataLoader(
    test_dataset, 
    sampler=SequentialSampler(test_dataset), 
    batch_size=batch_size 
)

In [ ]:
from transformers import BertForSequenceClassification, AdamW, BertConfig, logging
logging.set_verbosity_error()


model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2, 
    output_attentions = False, 
    output_hidden_states = False,
)

model.cuda()

In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(),
                  lr = 2e-5, 
                  eps = 1e-8 
                )



epochs = 2
total_steps = len(train_dataloader) * epochs

# Create the learning rate scheduler.
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0,
                                            num_training_steps = total_steps)


In [ ]:
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train_epoch(model, train_dataloader, optimizer, scheduler, device):


    model.train()


    total_train_loss = 0

    for step, batch in enumerate(train_dataloader):
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)


        model.zero_grad()

        # forward pass
        result = model(b_input_ids, 
                       token_type_ids=None, 
                       attention_mask=b_input_mask, 
                       labels=b_labels,
                       return_dict=True)

        loss = result.loss
        logits = result.logits

        total_train_loss += loss.item()

        #  backward pass 
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        
        if (step + 1) % 1000 == 0:
            print(f"Batch {step + 1}/{len(train_dataloader)}: Loss = {loss.item():.4f}")

    avg_train_loss = total_train_loss / len(train_dataloader)
    
    return avg_train_loss

def validate_epoch(model, validation_dataloader, device):

    model.eval()


    total_eval_accuracy = 0
    total_eval_loss = 0


    for batch in validation_dataloader:
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        with torch.no_grad():
            result = model(b_input_ids, 
                           token_type_ids=None, 
                           attention_mask=b_input_mask,
                           labels=b_labels,
                           return_dict=True)

        loss = result.loss
        logits = result.logits

        total_eval_loss += loss.item()

        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()

        total_eval_accuracy += flat_accuracy(logits, label_ids)

    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    
    return avg_val_loss, avg_val_accuracy


def save_best_model(model, output_dir, best_val_loss, avg_val_loss):

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        print(f"  New best validation loss: {best_val_loss:.2f}. Saving model...")
        # Save the model and tokenizer
        model_to_save = model.module if hasattr(model, 'module') else model
        model_to_save.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir) 
    return best_val_loss

## Model Training

In [ ]:
def train_model(model, train_dataloader, validation_dataloader, optimizer, scheduler, epochs, device, output_dir):
    training_stats = []
    best_val_loss = float('inf')  # Initialize the best validation loss to infinity

    for epoch_i in range(epochs):
        print(f"\n======== Epoch {epoch_i + 1} / {epochs} ========")
        print("Training...")

        avg_train_loss = train_epoch(model, train_dataloader, optimizer, scheduler, device)
        print(f"\n  Average training loss: {avg_train_loss:.2f}")

        print("\nRunning Validation...")
        avg_val_loss, avg_val_accuracy = validate_epoch(model, validation_dataloader, device)
        print(f"  Accuracy: {avg_val_accuracy:.2f}")
        print(f"  Validation Loss: {avg_val_loss:.2f}")

        # Record statistics for this epoch
        training_stats.append({
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
        })


        best_val_loss = save_best_model(model, output_dir, best_val_loss, avg_val_loss)

    print("\nTraining complete!")
    return training_stats


In [ ]:
output_dir = '/kaggle/working/best_model_save/'
training_stats = train_model(model, train_dataloader, validation_dataloader, optimizer, scheduler, epochs, device, output_dir)

## Evaluate on the test set


In [ ]:
def predict_and_evaluate(model, test_dataloader, device):

    model.eval()

    total_test_loss = 0
    total_test_accuracy = 0
    predictions = []
    true_labels = []


    with torch.no_grad():
        for batch in test_dataloader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)
            b_labels = batch[2].to(device)


            outputs = model(b_input_ids, 
                            token_type_ids=None, 
                            attention_mask=b_input_mask,
                            labels=b_labels,
                            return_dict=True)

            loss = outputs.loss
            logits = outputs.logits

            total_test_loss += loss.item()

            logits = logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()


            total_test_accuracy += flat_accuracy(logits, label_ids)


            pred_flat = np.argmax(logits, axis=1).flatten()
            predictions.extend(pred_flat)
            true_labels.extend(label_ids.flatten())

    # Calculate average loss and accuracy
    avg_test_loss = total_test_loss / len(test_dataloader)
    avg_test_accuracy = total_test_accuracy / len(test_dataloader)

    return predictions, true_labels, avg_test_loss, avg_test_accuracy


## Load Best Model 

In [ ]:

my_model = BertForSequenceClassification.from_pretrained(output_dir)
tokenizer = BertTokenizer.from_pretrained(output_dir)


my_model.to(device)


In [ ]:
# Evaluate on the test set
predictions, true_labels, test_loss, test_accuracy = predict_and_evaluate(my_model, test_dataloader, device)

print(f"Test Accuracy: {test_accuracy:.2f}")
print(f"Test Loss: {test_loss:.2f}")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(predictions, true_labels, class_names):

    cm = confusion_matrix(true_labels, predictions)
    
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    
    # Plot the confusion matrix
    fig, ax = plt.subplots(figsize=(8, 8))
    disp.plot(cmap='Blues', ax=ax, values_format='d')  # 'd' for displaying raw counts
    plt.title("Confusion Matrix (Raw Counts)")
    plt.show()

    report = classification_report(true_labels, predictions, target_names=class_names)
    print("Classification Report:\n")
    print(report)

class_names = ['Negative', 'Positive']  

plot_confusion_matrix(predictions, true_labels, class_names)


In [ ]:

def process_and_predict_sentence(sentence, model, tokenizer, device='cuda'):

    encoded_dict = tokenizer.encode_plus(
        sentence,                     
        add_special_tokens=True,     
        max_length=64,                
        pad_to_max_length=True,      
        truncation=True,              
        return_attention_mask=True,   
        return_tensors='pt'           
    )


    input_ids = encoded_dict['input_ids'].to(device)
    attention_mask = encoded_dict['attention_mask'].to(device)


    model.eval()


    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits

    probabilities = torch.softmax(logits, dim=1)



    return probabilities


## Test The Model

In [ ]:
test_sentence = "The movie was fantastic!"
prob = process_and_predict_sentence(test_sentence, model, tokenizer)

prob

- **Probability of Negative Class: 0.0013 (or 0.13%)**
- **Probability of Positive Class: 0.9987 (or 99.87%)**

In [ ]:
test_sentence = "I am feeling sad today."
prob = process_and_predict_sentence(test_sentence, model, tokenizer)

prob

- **Probability of Negative Class: 0.998 (or 99.8%)**
- **Probability of Positive Class: 0.0014 (or 0.14%)**